<style>
:root {
  --ink: #17202a;
  --muted: #64748b;
  --paper: #fbf7ef;
  --card: #fffaf0;
  --line: #e7dcc9;
  --navy: #11263f;
  --teal: #0f766e;
  --amber: #b45309;
  --rose: #be123c;
  --green: #15803d;
}
body, .jp-Notebook { background: var(--paper); color: var(--ink); }
.jp-Cell, .cell { margin-bottom: 1.35rem; }
h1, h2, h3 { color: var(--navy); letter-spacing: -0.03em; }
h1 { font-size: 2.5rem; margin-bottom: .35rem; }
h2 { font-size: 1.55rem; margin-top: 2.1rem; }
p, li { font-size: 1.02rem; line-height: 1.65; }
a { color: var(--teal); font-weight: 700; }
.jano-hero { background: linear-gradient(135deg, #fff7ed 0%, #f0fdfa 54%, #e0f2fe 100%); border: 1px solid var(--line); border-radius: 28px; padding: 2rem; box-shadow: 0 22px 70px rgba(17, 38, 63, .10); }
.jano-eyebrow { color: var(--amber); text-transform: uppercase; font-size: .75rem; font-weight: 800; letter-spacing: .16em; }
.jano-lead { color: #334155; max-width: 900px; font-size: 1.08rem; }
.jano-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(170px, 1fr)); gap: 14px; margin: 1rem 0 1.25rem; }
.jano-card { background: var(--card); border: 1px solid var(--line); border-radius: 18px; padding: 1rem; box-shadow: 0 10px 30px rgba(17, 38, 63, .07); }
.jano-card strong { display: block; color: var(--muted); font-size: .72rem; text-transform: uppercase; letter-spacing: .12em; margin-bottom: .35rem; }
.jano-card span { color: var(--navy); font-size: 1.35rem; font-weight: 800; }
.jano-note { border-left: 5px solid var(--teal); background: #ecfeff; border-radius: 16px; padding: 1rem 1.2rem; color: #164e63; }
.jano-warning { border-left-color: var(--amber); background: #fffbeb; color: #78350f; }
.jano-section { background: rgba(255,255,255,.58); border: 1px solid var(--line); border-radius: 24px; padding: 1.25rem 1.35rem; margin: 1.2rem 0; }
.jano-table table, table.dataframe { border-collapse: collapse; width: 100%; font-size: .9rem; background: white; border-radius: 16px; overflow: hidden; box-shadow: 0 12px 35px rgba(17, 38, 63, .08); }
.jano-table th, .jano-table td, table.dataframe th, table.dataframe td { border-bottom: 1px solid #edf2f7; padding: .68rem .72rem; text-align: right; }
.jano-table th, table.dataframe th { background: #11263f; color: #f8fafc; font-weight: 700; }
.jano-table tr:nth-child(even) td, table.dataframe tr:nth-child(even) td { background: #f8fafc; }
.jano-table .idx { color: #64748b; font-weight: 700; }
.output_area, .jp-OutputArea-output { margin-top: .65rem; }
pre, code { font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace; }
pre.jano-code { background: #0f172a; color: #d1fae5; border-radius: 18px; padding: 1rem; overflow-x: auto; font-size: .83rem; line-height: 1.55; border: 1px solid #1e293b; }
details.jano-code-wrap { margin: .65rem 0; }
details.jano-code-wrap summary { cursor: pointer; color: var(--teal); font-weight: 800; font-size: .9rem; }
.jano-pill { display: inline-flex; align-items: center; gap: .35rem; border-radius: 999px; background: #ccfbf1; color: #134e4a; padding: .32rem .7rem; font-weight: 800; font-size: .82rem; margin: .2rem .25rem .2rem 0; }
.jano-pill.amber { background: #fef3c7; color: #92400e; }
.jano-pill.rose { background: #ffe4e6; color: #9f1239; }
</style>

<div class="jano-hero">
  <div class="jano-eyebrow">Real dataset walkthrough</div>
  <h1>Jano on BTS Airline On-Time Performance</h1>
  <p class="jano-lead">This notebook uses a real January 2024 extract from the U.S. Bureau of Transportation Statistics to show how Jano plans leakage-aware temporal simulations before model evaluation.</p>
  <span class="jano-pill">walk-forward plan</span>
  <span class="jano-pill amber">calendar-aligned days</span>
  <span class="jano-pill rose">leakage-aware timestamps</span>
</div>

Official source:

- BTS On-Time Performance overview: https://www.transtats.bts.gov/ontime/
- BTS table download page: https://www.transtats.bts.gov/DL_SelectFields.aspx?QO_fu146_anzr=b0-gvzr&gnoyr_VQ=FGJ
- Direct January 2024 PREZIP file used here: https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip

<div class="jano-note jano-warning">The BTS zip/CSV is intentionally kept out of git because it is a large external dataset. The repository ignores <code>data/</code>, so this notebook is versioned but the downloaded dataset is not.</div>


In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import zipfile

import numpy as np
import pandas as pd

from jano import (
    DriftMonitoringPolicy,
    TemporalPartitionSpec,
    TemporalSemanticsSpec,
    TrainHistoryPolicy,
    WalkForwardPolicy,
)

BTS_ZIP_URL = "https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip"
DATA_PATH = Path("data/bts/bts_ontime_2024_01.zip")
if not DATA_PATH.exists():
    DATA_PATH = Path("../data/bts/bts_ontime_2024_01.zip")

if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(BTS_ZIP_URL, DATA_PATH)

DATA_PATH

'data/bts/bts_ontime_2024_01.zip'

## 1. Load a Real Operational Slice

The raw monthly BTS file is large enough to make planning realistic. We keep only ATL departures and a compact set of columns so the notebook stays fast while preserving the temporal problem.


In [2]:
flights = load_bts_sample(DATA_PATH)
flights[preview_columns].head()

scheduled_departure_at,actual_arrival_at,Origin,Dest,Reporting_Airline,is_arrival_delayed
2024-01-01 05:21,2024-01-01 07:22,ATL,FLL,NK,0
2024-01-01 05:35,2024-01-01 07:36,ATL,LGA,NK,0
2024-01-01 05:36,2024-01-01 07:13,ATL,MIA,AA,0
2024-01-01 05:47,2024-01-01 07:34,ATL,DFW,AA,0
2024-01-01 05:50,2024-01-01 07:34,ATL,MCO,F9,0


## 2. Define Leakage-Aware Time Semantics

The operational timeline is `scheduled_departure_at`. Training eligibility uses `actual_arrival_at`, because the supervised target is only known once the flight has arrived. Test eligibility stays anchored on scheduled departure.


In [3]:
semantics = TemporalSemanticsSpec(
    timeline_col="scheduled_departure_at",
    segment_time_cols={
        "train": "actual_arrival_at",
        "test": "scheduled_departure_at",
    },
)

walk_forward = WalkForwardPolicy(
    time_col=semantics,
    partition=TemporalPartitionSpec(
        layout="train_test",
        train_size="7D",
        test_size="1D",
        calendar_frequency="D",
    ),
    step="1D",
    strategy="rolling",
    max_folds=8,
)

<HTML output>

## 3. Inspect the Plan Before Materializing Folds

A [`plan()`](https://marmurar.github.io/jano/concepts.html#planning-before-materialization) is the precomputed geometry of a simulation. It shows the temporal intent before any train/test slices are materialized or any model is trained.


In [4]:
plan = walk_forward.plan(flights, title="ATL January 2024 walk-forward plan")
plan_df = plan.to_frame()
plan_df[["iteration", "train_start", "train_end", "test_start", "test_end", "train_rows", "test_rows"]]

iteration,train_start,train_end,test_start,test_end,train_rows,test_rows
0,2024-01-01,2024-01-08,2024-01-08,2024-01-09,5720,829
1,2024-01-02,2024-01-09,2024-01-09,2024-01-10,5829,792
2,2024-01-03,2024-01-10,2024-01-10,2024-01-11,5754,817
3,2024-01-04,2024-01-11,2024-01-11,2024-01-12,5738,853
4,2024-01-05,2024-01-12,2024-01-12,2024-01-13,5773,827
5,2024-01-06,2024-01-13,2024-01-13,2024-01-14,5737,759
6,2024-01-07,2024-01-14,2024-01-14,2024-01-15,5725,780
7,2024-01-08,2024-01-15,2024-01-15,2024-01-16,5655,842


## 4. Remove a Special Date Before Execution

Because the plan is inspectable, we can exclude folds before materialization. Here we remove folds whose train segment overlaps Martin Luther King Jr. Day.


In [5]:
filtered_plan = plan.exclude_windows(train=[("2024-01-15", "2024-01-16")])
filtered_plan.to_frame()[["iteration", "train_start", "train_end", "test_start", "test_end"]]

iteration,train_start,train_end,test_start,test_end
0,2024-01-01,2024-01-08,2024-01-08,2024-01-09
1,2024-01-02,2024-01-09,2024-01-09,2024-01-10
2,2024-01-03,2024-01-10,2024-01-10,2024-01-11
3,2024-01-04,2024-01-11,2024-01-11,2024-01-12
4,2024-01-05,2024-01-12,2024-01-12,2024-01-13
5,2024-01-06,2024-01-13,2024-01-13,2024-01-14
6,2024-01-07,2024-01-14,2024-01-14,2024-01-15
7,2024-01-08,2024-01-15,2024-01-15,2024-01-16


## 5. Materialize Only the Folds We Want

Materialization turns the plan into actual fold objects and row indices. This is the point where a training loop or downstream evaluation can consume the splits.


In [6]:
simulation_result = filtered_plan.materialize()
simulation_result.total_folds, simulation_result.to_frame().head()

fold,simulation_start,simulation_end,train_start,train_end,train_rows,test_start,test_end,test_rows
0,2024-01-01,2024-01-09,2024-01-01,2024-01-08,5720,2024-01-08,2024-01-09,829
1,2024-01-02,2024-01-10,2024-01-02,2024-01-09,5829,2024-01-09,2024-01-10,792
2,2024-01-03,2024-01-11,2024-01-03,2024-01-10,5754,2024-01-10,2024-01-11,817
3,2024-01-04,2024-01-12,2024-01-04,2024-01-11,5738,2024-01-11,2024-01-12,853
4,2024-01-05,2024-01-13,2024-01-05,2024-01-12,5773,2024-01-12,2024-01-13,827


## 6. Model Hypothesis: How Much Train History Is Enough?

The model here is intentionally simple. The point is not to build a production-grade delay classifier, but to show how a temporal policy can test a hypothesis on real time-correlated data.


In [7]:
class SimpleDelayRateClassifier:
    def fit(self, X: pd.DataFrame, y: pd.Series):
        keys = self._keys(X)
        grouped = pd.DataFrame({"key": keys, "target": y.to_numpy()}).groupby("key")["target"].mean()
        self.rate_by_key = grouped.to_dict()
        self.global_rate = float(np.mean(y.to_numpy()))
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        keys = self._keys(X)
        return np.array([1 if self.rate_by_key.get(key, self.global_rate) >= 0.5 else 0 for key in keys])

    @staticmethod
    def _keys(X: pd.DataFrame) -> pd.Series:
        return X[["Origin", "Dest", "Reporting_Airline", "DayOfWeek", "scheduled_dep_hour"]].astype(str).agg("|".join, axis=1)

feature_cols = ["Origin", "Dest", "Reporting_Airline", "DayOfWeek", "scheduled_dep_hour", "Distance"]
model = SimpleDelayRateClassifier()

<HTML output>

In [8]:
train_history = TrainHistoryPolicy(
    semantics,
    cutoff=plan_df.iloc[3]["train_end"],
    train_sizes=["2D", "4D", "6D", "7D"],
    test_size="1D",
)

train_history_result = train_history.evaluate(
    flights,
    model=model,
    target_col="is_arrival_delayed",
    feature_cols=feature_cols,
    metrics="accuracy",
)

train_history_result.to_frame()

variant,train_size,train_start,train_end,test_start,test_end,train_rows,test_rows,accuracy
0,2 days 00:00:00,2024-01-09,2024-01-11,2024-01-11,2024-01-12,1615,853,0.856
1,4 days 00:00:00,2024-01-07,2024-01-11,2024-01-11,2024-01-12,3279,853,0.856
2,6 days 00:00:00,2024-01-05,2024-01-11,2024-01-11,2024-01-12,4908,853,0.853
3,7 days 00:00:00,2024-01-04,2024-01-11,2024-01-11,2024-01-12,5738,853,0.831


## 7. Drift Monitoring: What Happens If Train Stays Fixed?

This policy freezes train and moves test forward. It answers a different operational question: how long can the current model remain behind an API before performance starts to decay?


In [9]:
drift_monitor = DriftMonitoringPolicy(
    semantics,
    cutoff=plan_df.iloc[3]["train_end"],
    train_size="7D",
    test_size="1D",
    step="1D",
    max_windows=6,
)

drift_result = drift_monitor.evaluate(
    flights,
    model=model,
    target_col="is_arrival_delayed",
    feature_cols=feature_cols,
    metrics="accuracy",
)

drift_result.to_frame()

window,train_size,train_start,train_end,test_start,test_end,train_rows,test_rows,accuracy
0,7 days 00:00:00,2024-01-04,2024-01-11,2024-01-11,2024-01-12,5738,853,0.831
1,7 days 00:00:00,2024-01-04,2024-01-11,2024-01-12,2024-01-13,5738,827,0.637
2,7 days 00:00:00,2024-01-04,2024-01-11,2024-01-13,2024-01-14,5738,759,0.689
3,7 days 00:00:00,2024-01-04,2024-01-11,2024-01-14,2024-01-15,5738,780,0.723
4,7 days 00:00:00,2024-01-04,2024-01-11,2024-01-15,2024-01-16,5738,842,0.595
5,7 days 00:00:00,2024-01-04,2024-01-11,2024-01-16,2024-01-17,5738,824,0.515


## Notes

- The notebook prioritizes temporal geometry and leakage-aware validation over model sophistication.
- `calendar_frequency="D"` aligns folds to complete calendar days.
- The plan-first workflow makes the simulation auditable before folds are materialized.
- Replace the toy classifier with your own model pipeline while keeping the same temporal setup.
